# Pretrain the ball tracker on the TrackNetV2 dataset

Pretrains the heatmap tracker on the public TrackNetV2 badminton (shuttlecock) dataset so it learns to follow a tiny fast ball; you then fine-tune on your own cricket footage.

**Research / non-commercial use only.** A model trained on this must not ship in a commercial product; the shipped model is trained on your own data.

Dataset: 26 broadcast matches, 1280x720, 30 fps, ~78,200 frames. **You download it yourself** (steps below); this notebook does not fetch it.

In [ ]:
!git clone --depth 1 https://github.com/LK-maker-007/Snarl.git
%cd Snarl/ml
!pip install -q -e '.[convert]'

## 1. Get the dataset

1. Download the TrackNetV2 shuttlecock dataset from the official NYCU link:
   `https://nycu1-my.sharepoint.com/:u:/g/personal/tik_m365_nycu_edu_tw/EWisYhAiai9Ju7L-tQp0ykEBZJd9VQkKqsFrjcqqYIDP-g?e=S0AB1Z`
   (SharePoint needs a browser, so download to your computer, then upload the zip to your Google Drive — e.g. `MyDrive/TrackNetV2.zip`.)
2. The extracted layout is `profession_dataset/match[N]/frame/<rally>/0.png...` with labels in `match[N]/ball_trajectory/<rally>_ball.csv` (and an `amateur_dataset/` with the same shape).

Mount Drive and unzip (edit the zip path to match yours):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/TrackNetV2.zip'   # <-- edit to your uploaded path
!mkdir -p /content/tracknet && unzip -q -o "{ZIP}" -d /content/tracknet
!find /content/tracknet -maxdepth 2 -type d | head -20

## 2. Arrange into clip directories

Turns the TrackNetV2 layout into one clip dir per rally (frames symlinked + a converted `labels.csv`) that the loader reads. Point `TRACKNET_SRC` at the extracted root.

In [ ]:
TRACKNET_SRC = '/content/tracknet'   # extracted dataset root (contains profession_dataset/ etc.)
DATA_ROOT = '/content/data'          # output clip dirs for training
!python -m snarl_ml.arrange '{TRACKNET_SRC}' '{DATA_ROOT}'

## 3. Pretrain

Trains the tracker, reporting per-epoch loss and decoded pixel error, and saves the best checkpoint to `/content/pretrained.pt`. (Use a GPU runtime: Runtime -> Change runtime type -> GPU.)

In [ ]:
!python -m snarl_ml.train --image-dir '{DATA_ROOT}' --image-glob '*.png' --epochs 30 --out /content/pretrained.pt

## Next

`/content/pretrained.pt` is a research pretrained tracker — **download it before the session ends.** Fine-tune it on your own consented cricket clips (see `ml/DATA_COLLECTION.md`):

```
python -m snarl_ml.train --image-dir <cricket-clips> --init-from pretrained.pt --epochs 50
```